In [ ]:
queryde=QueryDecomposer(llm=llm)
hyde_gen = HyDEGenerator(llm=llm, embed=embed)
from sentence_transformers import CrossEncoder
cross_encoder = CrossEncoder("BAAI/bge-reranker-v2-m3",device="cpu")
@tool
def search_fpt_policy(query: str) -> str:
    """
    Hãy sử dụng công cụ này BẮT BUỘC khi bạn cần tìm kiếm thông tin về quy định, 
    chính sách, nội quy, giờ giấc, lương thưởng hoặc bất kỳ vấn đề nhân sự nào của FPT.
    """
    print(f"\n[AGENT TOOL] Đang xử lý câu hỏi: '{query}'")
    sub_qs = queryde.decompose(query)
    print(sub_qs)
    all_contexts = []
    for su in sub_qs:
        # hyx=hyde_gen.generate_hypothetical_answer(su)
        doc=ensenble_retriever.invoke(su)
        all_contexts.extend(doc)  
    unique_contents = list(set([doc.page_content for doc in all_contexts]))
    if len(unique_contents) <=3:
        final_context = "\n\n======\n\n".join(unique_contents)
        return final_context
    ranks=cross_encoder.rank(query, unique_contents)
    top_contexts = []
    for rank in ranks:
        doc_index = rank['corpus_id']
        doc_content = unique_contents[doc_index]
        
        # BỘ LỌC RÁC: Xóa khoảng trắng 2 đầu, nếu text ngắn hơn 15 ký tự -> Bỏ qua
        if len(doc_content.strip()) > 15:
            print(f"\n[DEBUG] Rank {rank['score']}: {doc_content.strip()}\n")
            top_contexts.append(doc_content)
            
        # Chỉ lấy đủ 3 chunks tốt nhất thì dừng
        if len(top_contexts) == 3:
            break
    final_context = "\n\n======\n\n".join(top_contexts)
    return final_context
    
# def policy_decision(context: str, question: str) -> str:
#     """
#     Đưa ra quyết định dựa trên policy.
#     """
#     prompt = f"""
#     Policy:
#     {context}

#     Question:
#     {question}

#     Trả lời có/không + giải thích.
#     """
#     return llm.invoke(prompt).content
tools = [search_fpt_policy]

Note: you may need to restart the kernel to use updated packages.


In [38]:
import os
from langchain_community.document_loaders import PyPDFLoader,TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.vectorstores import Chroma
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain_ollama import OllamaEmbeddings
from langchain_ollama import OllamaLLM
from langchain_ollama import ChatOllama
import hashlib
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_experimental.text_splitter import SemanticChunker
from sentence_transformers import CrossEncoder


In [39]:
rm -rf ./chroma_db

In [40]:
# Cập nhật lại Cell 21 của em
import chromadb

# Khởi tạo client tĩnh
client = chromadb.Client()

# Chủ động xóa collection cũ nếu nó đang tồn tại trong RAM
try:
    client.delete_collection("fpt_policy")
    print("[LOG] Đã dọn sạch Database cũ!")
except Exception:
    pass

[LOG] Đã dọn sạch Database cũ!


In [41]:
loader = TextLoader("fpt_policy.txt", encoding="utf-8")
documents = loader.load()

========================sematicchuck====================

In [42]:
embed = OllamaEmbeddings(model="qwen3-embedding")
# text_splitter = SemanticChunker(
#     embed, 
#     breakpoint_threshold_type="percentile",
#     breakpoint_threshold_amount=75
# )
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,          # Độ dài lý tưởng để Reranker đọc không bị hụt hơi
    chunk_overlap=50,        # Giữ lại một chút ngữ cảnh nối câu
    separators=["\n\n---", "\n\n", "\n", " ", ""] # Ưu tiên cắt ở các vạch phân cách trước
)
texts = text_splitter.split_documents(documents)

# Xem thử 3 mảnh cắt đầu tiên trông như thế nào
for i in range(3):
    print(f"--- Chunk {i+1} ---")
    print(texts[i].page_content)

--- Chunk 1 ---
FPT INTERNAL POLICY (EXTENDED VERSION)

---

Document ID: FPT-HR-001
Effective Date: 01/01/2024
Scope: Applies to all full-time employees at FPT and its affiliated units.
--- Chunk 2 ---
---

1. WORKING HOURS

* Standard working time: 8 hours/day, Monday to Friday.
* Morning: 08:30 - 12:00
* Lunch break: 12:00 - 13:30
* Afternoon: 13:30 - 17:30

Additional Rules:

* Employees must be present at least 5 minutes before working hours.
* Flexible working hours may be applied depending on department approval.
* Remote work must be approved by direct manager.
--- Chunk 3 ---
---

2. ATTENDANCE REGULATION

* Employees must check-in/check-out via myFPT system:

  * Facial recognition
  * Employee card
* Attendance data is the official basis for payroll and discipline.

Late/Early Leave:

* Late after 08:30 or leaving before 17:30 without approval is considered a violation.

Penalty:


============================end=======================

=================================hnsw indexing=========================

In [43]:
documents = loader.load()
doc_ids = [f"{i}_" + hashlib.md5(doc.page_content.encode('utf-8')).hexdigest() for i, doc in enumerate(texts)]
collection_metadata={
    "hnsw:space":"cosine",
    "hnsw:M":64,
    "hnsw:construction_ef":128,
    "hnsw:search_ef":128
}
llm = ChatOllama(
    model="llama3.1", 
    temperature=0, 
)
vectorstore = Chroma.from_documents(documents=texts, 
                                    embedding=embed,
                                    ids=doc_ids, 
                                    collection_name="fpt_policy", 
                                    collection_metadata=collection_metadata)
"""
Truyền ids vào from_documents để tránh trùng lặp dữ liệu khi chạy lại script nhiều lần. Nếu không truyền ids, mỗi lần chạy sẽ tạo ra các chunk mới với ID mới, dẫn đến việc lưu trữ nhiều bản sao giống hệt nhau trong cơ sở dữ liệu
    doc_ids = generate_chunk_ids(texts)

# Truyền ids vào from_documents
vectorstore = Chroma.from_documents(
    documents=texts, 
    embedding=embed, 
    ids=doc_ids, # <--- Điểm chốt 
    collection_name="fpt_policy",
    persist_directory=CHROMA_PERSIST_DIR
)
"""

'\nTruyền ids vào from_documents để tránh trùng lặp dữ liệu khi chạy lại script nhiều lần. Nếu không truyền ids, mỗi lần chạy sẽ tạo ra các chunk mới với ID mới, dẫn đến việc lưu trữ nhiều bản sao giống hệt nhau trong cơ sở dữ liệu\n    doc_ids = generate_chunk_ids(texts)\n\n# Truyền ids vào from_documents\nvectorstore = Chroma.from_documents(\n    documents=texts, \n    embedding=embed, \n    ids=doc_ids, # <--- Điểm chốt \n    collection_name="fpt_policy",\n    persist_directory=CHROMA_PERSIST_DIR\n)\n'

=============================end========================

=========================hybidsearch===================

In [44]:
def create_retriever(texts=texts, vectorstore=vectorstore):
    bm25_retriever=BM25Retriever.from_documents(texts)
    bm25_retriever.k=3
    ensenble_retriever = EnsembleRetriever(retrievers=[vectorstore.as_retriever(search_kwargs={"k": 5}), bm25_retriever], 
                                weights=[0.5, 0.5])
    return ensenble_retriever #tra ve 3 chunk tu vectorstore va 3 chunk tu bm25 retriever

In [45]:
# retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [46]:
ensenble_retriever = create_retriever(texts=texts, vectorstore=vectorstore)

============================end=========================

================_HyDE&decomposer_query code=====================

In [47]:
class HyDEGenerator:
    def __init__(self, llm, embed):
        self.llm = llm
        self.embed = embed
    def generate_hypothetical_answer(self,query: str) -> str:
        """
        Sinh ra đoạn văn bản giả định chứa từ khóa chuyên môn.
        """
        system_prompt ="""
            Bạn là một chuyên gia Nhân sự (HR). Người dùng đang hỏi một câu rất ngắn hoặc mơ hồ về chính sách công ty.
            Nhiệm vụ của bạn: Hãy viết một đoạn văn (khoảng 3-4 câu) GIẢ ĐỊNH trả lời cho câu hỏi đó. 
            Không cần thông tin số liệu chính xác, nhưng PHẢI dùng các thuật ngữ chuyên ngành HR, quy trình công ty, văn bản pháp lý liên quan.
            Không mở bài hay kết luận, chỉ viết phần nội dung chính.
        """
        hypothetical_answer= self.llm.invoke(system_prompt + "\n\nUser question: " + query).content
        print(f"Hypothetical answer generated for query '{query}':\n{hypothetical_answer}\n")
        return hypothetical_answer
    def get_embeddings(self,query:str)-> list:
        heypothetical_answer = self.generate_hypothetical_answer(query)
        embedding = self.embed.embed_query(heypothetical_answer)
        return embedding


In [48]:
import json
class QueryDecomposer:
    def __init__(self, llm):
        self.llm = llm
    def decompose(self, query: str) -> list[str]:
        """
        Tách câu hỏi phức tạp thành danh sách các câu hỏi con độc lập.
        """
        system_prompt = """
        Người dùng đang đặt một câu hỏi phức tạp yêu cầu thông tin từ nhiều khía cạnh hoặc so sánh.
        Hãy phân rã câu hỏi này thành một danh sách các câu hỏi con đơn giản hơn.
        Nguyên tắc quan trọng: Mỗi câu hỏi con phải TỰ ĐỨNG ĐỘC LẬP (phải giữ nguyên ngữ cảnh, không dùng đại từ thay thế như "nó", "chính sách đó").
        CHỈ trả về mảng JSON, tuyệt đối KHÔNG giải thích, KHÔNG bọc bằng markdown, KHÔNG thêm bất kỳ từ ngữ nào khác.
        Chỉ trả về JSON định dạng mảng các chuỗi: ["câu hỏi 1", "câu hỏi 2"]
        """
        response = self.llm.invoke(system_prompt + "\n\nUser question: " + query).content
        print(f"\n[DEBUG RAW LLM DECOMPOSE] {response}\n")
        try:
            sub_questions = json.loads(response)
            if isinstance(sub_questions, list) and all(isinstance(q, str) for q in sub_questions):
                return sub_questions
            else:
                raise ValueError("Response is not a list of strings")
        except json.JSONDecodeError:
            raise ValueError("Response is not valid JSON")
    def get_context(self, query: str) -> str:
        """
        Lấy ngữ cảnh liên quan cho câu hỏi con bằng cách sử dụng retriever.
        """
        sub_questions = self.decompose(query)
        contexts = []
        for sub_q in sub_questions:
            results = ensenble_retriever.invoke(sub_q)
            context = "\n\n---\n\n".join([result.page_content for result in results])
            contexts.append(context)
        return contexts

In [49]:
# queryde=QueryDecomposer(llm=llm)
# hyde_gen = HyDEGenerator(llm=llm, embed=embed)

# sub_qs = queryde.decompose("Quy định về làm thêm giờ của FPT là gì? Có được trả lương làm thêm giờ không?")
# print(sub_qs)
# all_contexts = []
# for su in sub_qs:
#     hyx=hyde_gen.generate_hypothetical_answer(su)
#     doc=ensenble_retriever.invoke(hyx)
#     all_contexts.extend(doc)  
# unique_contents = list(set([doc.page_content for doc in all_contexts]))
# final_context = "\n\n======\n\n".join(unique_contents)
# print("[LOG] Final Context Length:", len(final_context))

========================end========================

In [50]:
queryde=QueryDecomposer(llm=llm)
hyde_gen = HyDEGenerator(llm=llm, embed=embed)
from sentence_transformers import CrossEncoder
cross_encoder = CrossEncoder("cross-encoder/mmarco-mMiniLMv2-L12-H384-v1",device="cpu")
@tool
def search_fpt_policy(query: str) -> str:
    """
    Hãy sử dụng công cụ này BẮT BUỘC khi bạn cần tìm kiếm thông tin về quy định, 
    chính sách, nội quy, giờ giấc, lương thưởng hoặc bất kỳ vấn đề nhân sự nào của FPT.
    """
    print(f"\n[AGENT TOOL] Đang xử lý câu hỏi: '{query}'")
    sub_qs = queryde.decompose(query)
    print(sub_qs)
    all_contexts = []
    for su in sub_qs:
        # hyx=hyde_gen.generate_hypothetical_answer(su)
        doc=ensenble_retriever.invoke(su)
        all_contexts.extend(doc)  
    unique_contents = list(set([doc.page_content for doc in all_contexts]))
    if len(unique_contents) <=3:
        final_context = "\n\n======\n\n".join(unique_contents)
        return final_context
    ranks=cross_encoder.rank(query, unique_contents)
    top_contexts = []
    for rank in ranks:
        doc_index = rank['corpus_id']
        doc_content = unique_contents[doc_index]
        
        # BỘ LỌC RÁC: Xóa khoảng trắng 2 đầu, nếu text ngắn hơn 15 ký tự -> Bỏ qua
        if len(doc_content.strip()) > 15:
            print(f"\n[DEBUG] Rank {rank['score']}: {doc_content.strip()}\n")
            top_contexts.append(doc_content)
            
        # Chỉ lấy đủ 3 chunks tốt nhất thì dừng
        if len(top_contexts) == 3:
            break
    final_context = "\n\n======\n\n".join(top_contexts)
    return final_context
    
# def policy_decision(context: str, question: str) -> str:
#     """
#     Đưa ra quyết định dựa trên policy.
#     """
#     prompt = f"""
#     Policy:
#     {context}

#     Question:
#     {question}

#     Trả lời có/không + giải thích.
#     """
#     return llm.invoke(prompt).content
tools = [search_fpt_policy]

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 12017.38it/s]


In [51]:
# import torchresearch-llm/example/HungVX/week3/notebooks/RAG_new copy 2.ipynb
# import gc

# # 1. Xóa biến chứa mô hình (nếu nó đang tồn tại)
# try:
#     del cross_encoder
# except NameError:
#     pass

# # 2. Ép Python thu gom rác (những biến không còn dùng tới)
# gc.collect()

# # 3. Ép PyTorch dọn sạch bộ nhớ đệm (Cache) trên GPU
# torch.cuda.empty_cache()

# print("🧹 Đã dọn sạch CUDA VRAM!")

In [52]:
# search_fpt_policy("Quy định về làm thêm giờ của FPT là gì ? Có được trả lương làm thêm giờ không?")

In [53]:
SYSTEM_PROMPT = """Bạn là trợ lý Nhân sự (HR) mẫn cán của tập đoàn FPT.
Trả lời câu hỏi của nhân viên một cách thân thiện, ngắn gọn và mạch lạc. Chủ động xưng "mình" và gọi "bạn".

Tuyệt đối không tự bịa thông tin. Dựa 100% vào dữ liệu tìm được.

Chú ý: Nếu tin nhắn chỉ là giao tiếp (như Chào, Cảm ơn, Tạm biệt, ví dụ "Hello"), 
hãy phản hồi lịch sự ngắn gọn mà KHÔNG CẦN chạy tool tìm kiếm.Bạn CHỈ ĐƯỢC PHÉP trả lời dựa trên thông tin nhận được từ công cụ tìm kiếm (Observation). Nếu Observation không chứa thông tin liên quan đến câu hỏi, bạn bắt buộc phải trả lời: 'Tôi không tìm thấy thông tin này trong hệ thống chính sách'. Tuyệt đối không được tự suy đoán hoặc sử dụng kiến thức bên ngoài.

Nếu câu hỏi thuộc về công việc, chế độ hay chính sách, BẮT BUỘC dùng tool 'search_fpt_policy' để tìm thông tin trước khi trả lời.
"""
# mmodel = init_chat_model(model="openai:gpt-4.1-mini", temperature=0.0)
agent = create_agent(
    model=llm, 
    tools=tools, 
    system_prompt=SYSTEM_PROMPT).with_config(max_iterations=3)


In [ ]:
from langchain_core.documents import Document
import re, unicodedata

# ============================================================
# v2.5: Production-grade evaluation (FIXED Precision/Recall logic)
# ============================================================

def _normalize(text: str) -> str:
    text = unicodedata.normalize("NFKC", text)
    text = text.lower().strip()
    text = re.sub(r"[*\-\u2013\u2022\u00b7\u2192]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text

def _token_recall(chunk: str, truth: str) -> float:
    truth_tokens = set(_normalize(truth).split())
    chunk_tokens = set(_normalize(chunk).split())
    if not truth_tokens:
        return 0.0
    return len(truth_tokens & chunk_tokens) / len(truth_tokens)


def evaluate_retrieval(
    retrieved_docs: list,
    ground_truth_contents: list[str],
    match_mode: str = "substring",
    threshold: float = 0.5,
) -> dict:
    retrieved_texts = [
        doc.page_content if isinstance(doc, Document) else str(doc)
        for doc in retrieved_docs
    ]

    matched_gts = set()
    matched_chunks = set()
    matched_pairs = []

    for ret_idx, ret in enumerate(retrieved_texts):
        for gt_idx, truth in enumerate(ground_truth_contents):
            # 1 chunk có thể match nhiều GTs, 1 GT có thể nằm trong nhiều chunks
            is_match = False
            score = 0.0

            if match_mode == "substring":
                is_match = _normalize(truth) in _normalize(ret)
            elif match_mode == "token_recall":
                score = _token_recall(ret, truth)
                is_match = score >= threshold

            if is_match:
                matched_gts.add(gt_idx)
                matched_chunks.add(ret_idx)
                matched_pairs.append({
                    "retrieved_idx": ret_idx,
                    "truth_idx": gt_idx,
                    "score": round(_token_recall(ret, truth), 3),
                    "retrieved_snippet": " | ".join(ret.split("\n")[:3])[:120],
                    "truth_snippet": truth[:100],
                })

    # Tính toán cực chuẩn:
    # 1. Khía cạnh Chunk (để tính Precision & False Positives)
    # Bao nhiêu chunks mang thông tin hữu ích? (TP của Chunks)
    tp_chunks = len(matched_chunks)
    # Bao nhiêu chunks hoàn toàn vô dụng? (False Positives)
    false_positives = len(retrieved_texts) - tp_chunks

    # 2. Khía cạnh Ground Truth (để tính Recall & False Negatives)
    # Bao nhiêu thông tin cần thiết đã lấy được? (TP của GTs)
    tp_gts = len(matched_gts)
    # Bao nhiêu thông tin bị mất mát? (False Negatives)
    false_negatives = len(ground_truth_contents) - tp_gts

    # Precision: Trong số chunks lấy về, tỷ lệ % chunks HỮU ÍCH
    precision = tp_chunks / len(retrieved_texts) if retrieved_texts else 0.0
    # Recall: Trong số thông tin cần lấy, tỷ lệ % đã TÌM THẤY
    recall = tp_gts / len(ground_truth_contents) if ground_truth_contents else 0.0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

    return {
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "f1": round(f1, 4),
        "true_positives_chunks": tp_chunks,
        "true_positives_gts": tp_gts,
        "false_positives": false_positives,
        "false_negatives": false_negatives,
        "matched_pairs": matched_pairs,
        "n_retrieved": len(retrieved_texts),
        "n_ground_truth": len(ground_truth_contents),
    }

def print_eval_report(query: str, metrics: dict, show_matches: bool = True):
    print(f"\n{'='*65}")
    print(f"Query: {query}")
    print(f"{'='*65}")
    print(f"  Retrieved chunks   : {metrics['n_retrieved']}")
    print(f"  Ground truth items : {metrics['n_ground_truth']}")
    print(f"  {'--'*22}")
    print(f"  ✅ TP (GTs tìm thấy): {metrics['true_positives_gts']:>3} / {metrics['n_ground_truth']}")
    print(f"  ✅ TP (Chunks đúng):  {metrics['true_positives_chunks']:>3} / {metrics['n_retrieved']}")
    print(f"  ❌ FP (Chunks rác):   {metrics['false_positives']:>3}  -- chunks dư thừa")
    print(f"  ⚠️  FN (GT bị sót):    {metrics['false_negatives']:>3}  -- kiến thức bị sót")
    print(f"  {'--'*22}")
    print(f"  Precision : {metrics['precision']:.2%}")
    print(f"  Recall    : {metrics['recall']:.2%}")
    print(f"  F1-Score  : {metrics['f1']:.2%}")
    if show_matches and metrics.get("matched_pairs"):
        print(f"\n  Matched Pairs:")
        for p in metrics["matched_pairs"]:
            print(f"    [ret#{p['retrieved_idx']}] <-> [gt#{p['truth_idx']}]")
            print(f"      Chunk : {p['retrieved_snippet']!r}")
            print(f"      GT    : {p['truth_snippet']!r}")
    print(f"{'='*65}\n")


In [55]:
# question = "Quy định về làm thêm giờ của FPT là gì ? Có được trả lương làm thêm giờ không?"
# print(f"🧑 User hỏi: {question}\n")
# print("🔄 Agent đang suy nghĩ và tìm kiếm...\n")
# inputs = {
#         "messages": [
#             {"role": "user", "content": question}
#         ]
#     }
# for chunk in agent.stream(inputs):
#     node_name = list(chunk.keys())[0]
#     print(f"\n[Trạm: {node_name.upper()} vừa hoàn thành]")
    
#     new_message = chunk[node_name]["messages"][-1]
    
#     if new_message.type == "ai":
#         # Kiểm tra xem AI có gọi tool không
#         if new_message.tool_calls:
#             print(f"🤖 AI ra quyết định: Cần gọi công cụ '{new_message.tool_calls[0]['name']}'")
#             print(f"   Tham số truyền vào: {new_message.tool_calls[0]['args']}")
#         else:
#             # SỬA LỖI Ở ĐÂY: Xử lý an toàn cho nội dung trả về
#             content = new_message.content
#             if isinstance(content, list):
#                 # Dành cho model trả về dạng list of dicts
#                 text_output = content[0].get('text', '') if len(content) > 0 else ""
#             else:
#                 # Dành cho Ollama/các model trả về string thuần
#                 text_output = content
                
#             print(f"🤖 AI trả lời user: {text_output}")
            
#     elif new_message.type == "tool":
#         print(f"🛠️ Công cụ trả về dữ liệu (Observation):")
#         # In tối đa 150 ký tự để console đỡ rối
#         print(f"   '{str(new_message.content)[:150]}...'")

SyntaxError: invalid syntax (1455244962.py, line 1)